# 02 — Vintage Curve Analysis
**Credit Risk Vintage Curves & Survival Analysis — Lending Club (2007-2018)**

This notebook builds vintage (cohort) curves that track cumulative default rates by origination quarter across Months on Book (MOB). This is the gold-standard framework used by risk teams at every bank and lending fintech.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 150

In [ ]:
df = pd.read_csv('../data/clean_loans.csv', parse_dates=['issue_date', 'last_pymnt_date'])
print(f"Dataset: {len(df):,} loans")
print(f"Default rate: {df['is_default'].mean():.2%}")
print(f"MOB range: {df['mob'].min()} – {df['mob'].max()}")

## 1. Build Vintage Curves
Group loans by origination quarter (the "vintage") and calculate cumulative default rates at each Month on Book (MOB).

In [ ]:
# Build vintage curves: cumulative default rate at each MOB for each vintage quarter
max_mob = 36  # Track up to 36 months

vintages = []
for vintage, group in df.groupby('issue_quarter'):
    total_loans = len(group)
    if total_loans < 100:  # Skip small vintages
        continue
    for mob in range(1, max_mob + 1):
        defaults_by_mob = group[(group['mob'] <= mob) & (group['is_default'] == 1)].shape[0]
        vintages.append({
            'vintage_quarter': vintage,
            'mob': mob,
            'total_loans': total_loans,
            'cumulative_defaults': defaults_by_mob,
            'cumulative_default_rate': defaults_by_mob / total_loans
        })

vintage_df = pd.DataFrame(vintages)
print(f"Built vintage curves for {vintage_df['vintage_quarter'].nunique()} quarters")
vintage_df.head(10)

## 2. Plot Vintage Curves — Gradient Colors by Year

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

unique_vintages = sorted(vintage_df['vintage_quarter'].unique())
cmap = plt.cm.plasma(np.linspace(0, 1, len(unique_vintages)))

for i, v in enumerate(unique_vintages):
    subset = vintage_df[vintage_df['vintage_quarter'] == v]
    ax.plot(subset['mob'], subset['cumulative_default_rate'] * 100,
            color=cmap[i], linewidth=1.2, alpha=0.8, label=v)

ax.set_xlabel('Months on Book (MOB)', fontsize=12)
ax.set_ylabel('Cumulative Default Rate (%)', fontsize=12)
ax.set_title('Vintage Curve Analysis — Cumulative Default Rate by Origination Quarter',
             fontweight='bold', fontsize=14)
ax.grid(True, alpha=0.3)

# Place legend outside
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7, ncol=2, title='Vintage Quarter')

plt.tight_layout()
plt.savefig('../outputs/vintage_curves_all.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Vintage Curves — Selected Quarters (Cleaner View)

In [ ]:
selected = ['2008Q4', '2010Q1', '2012Q1', '2014Q1', '2016Q1', '2017Q1', '2018Q1']
selected = [v for v in selected if v in vintage_df['vintage_quarter'].unique()]

fig, ax = plt.subplots(figsize=(14, 7))
colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db', '#9b59b6', '#1abc9c']

for v, color in zip(selected, colors):
    subset = vintage_df[vintage_df['vintage_quarter'] == v]
    ax.plot(subset['mob'], subset['cumulative_default_rate'] * 100,
            color=color, linewidth=2.5, label=v, marker='o', markersize=3)

ax.set_xlabel('Months on Book (MOB)', fontsize=12)
ax.set_ylabel('Cumulative Default Rate (%)', fontsize=12)
ax.set_title('Vintage Curves — Selected Origination Quarters', fontweight='bold', fontsize=14)
ax.legend(title='Vintage', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/vintage_curves_selected.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Vintage × MOB Heatmap

In [ ]:
# Pivot: rows = vintage, columns = MOB, values = cumulative default rate
pivot = vintage_df.pivot_table(
    index='vintage_quarter', columns='mob',
    values='cumulative_default_rate'
).round(4)

# Show subset for readability (every other vintage, MOB 1-36 every 3)
display_vintages = sorted(pivot.index)[::2]  # every other quarter
display_mobs = list(range(1, 37, 3))  # MOB 1, 4, 7, ...

fig, ax = plt.subplots(figsize=(18, 10))
sns.heatmap(
    pivot.loc[display_vintages, display_mobs] * 100,
    annot=True, fmt='.1f', cmap='YlOrRd',
    linewidths=0.5, ax=ax, cbar_kws={'label': 'Cumulative Default Rate (%)'}
)
ax.set_title('Vintage × MOB Heatmap — Cumulative Default Rate (%)', fontweight='bold', fontsize=14)
ax.set_xlabel('Month on Book (MOB)')
ax.set_ylabel('Vintage Quarter')

plt.tight_layout()
plt.savefig('../outputs/vintage_mob_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. MOB at Which 50% of Eventual Defaults Occur

In [ ]:
# For each vintage, find the MOB at which 50% of total eventual defaults have occurred
mob_50pct = []
for vintage, group in vintage_df.groupby('vintage_quarter'):
    max_defaults = group['cumulative_defaults'].max()
    if max_defaults == 0:
        continue
    threshold = max_defaults * 0.5
    hit = group[group['cumulative_defaults'] >= threshold]
    if not hit.empty:
        mob_50pct.append({
            'vintage_quarter': vintage,
            'mob_at_50pct_loss': hit.iloc[0]['mob'],
            'total_defaults': max_defaults
        })

mob50_df = pd.DataFrame(mob_50pct)

fig, ax = plt.subplots(figsize=(14, 6))
ax.barh(mob50_df['vintage_quarter'], mob50_df['mob_at_50pct_loss'], color='#3498db')
ax.set_xlabel('MOB at 50% of Total Defaults')
ax.set_ylabel('Vintage Quarter')
ax.set_title('When Do Half of All Defaults Occur? (MOB at 50% Loss)', fontweight='bold')
ax.invert_yaxis()

for i, row in mob50_df.iterrows():
    ax.text(row['mob_at_50pct_loss'] + 0.3, i, str(int(row['mob_at_50pct_loss'])), va='center', fontsize=8)

plt.tight_layout()
plt.savefig('../outputs/mob_at_50pct_loss.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nAverage MOB at 50% default: {mob50_df['mob_at_50pct_loss'].mean():.1f} months")
print(f"Median MOB at 50% default: {mob50_df['mob_at_50pct_loss'].median():.0f} months")

## 6. Save Vintage Data for Power BI Dashboard

In [ ]:
vintage_df.to_csv('../outputs/vintage_curve_data.csv', index=False)
mob50_df.to_csv('../outputs/mob_50pct_data.csv', index=False)
print('✅ Exported vintage data for dashboard use')